<a href="https://colab.research.google.com/github/wwzong314/MpMTB/blob/main/04_colab_retrain_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Retrain v7 on Colab (Google Drive)

Runs the **real** `optimized_minimal_transfer_v7_raw_curve_msa_aux.py` rather than a
reimplementation. That script carries ~40 interacting hyperparameters and staged
auxiliary losses (raw-curve MSA aux from epoch 10, seq adapter from 20, full-bank loss
from 20, adaptive hard margin, top-1 focus from 80), so rewriting it would not be v7.

What this notebook adds is a **correctly built training CSV**: template-scoped joins,
the purity levers exposed, and the human plates included.

## Before you run

Put these on Drive under `DRIVE_ROOT` (paths configurable in the next cell):

| file | where from |
|---|---|
| `optimized_minimal_transfer_v7_raw_curve_msa_aux.py` | `~/Documents/AI_for_seq/` |
| `my_contrastive_model_withflanking0430.keras` | the 0430 base model |
| `organised/metadata.csv`, `organised/curves.npz` | output of notebook `01_organise_data` |

Then set **Runtime → Change runtime type → GPU**. On CPU this takes many hours.

If you have not run notebook 01 yet, set `USE_ORGANISED = False` to fall back to the
original training CSV and reproduce the published baseline (test R@1 = 0.18).

In [ ]:
# ----------------------------------------------------------------------------------
# CONFIG — edit this cell only
# ----------------------------------------------------------------------------------
DRIVE_ROOT = "/content/drive/Shareddrives/Kelvin Genomics share/traning_data"

V7_SCRIPT = f"{DRIVE_ROOT}/optimized_minimal_transfer_v7_raw_curve_msa_aux.py"
BASE_MODEL = f"{DRIVE_ROOT}/my_contrastive_model_withflanking0430.keras"
ORGANISED = f"{DRIVE_ROOT}/xing_0812_2026_base_0806/organised"                     # metadata.csv + curves.npz
FALLBACK_CSV = f"{DRIVE_ROOT}/training_filtered_no_replicate_sequences_no_resample_normalize_only.csv"

USE_ORGANISED = True          # False -> use FALLBACK_CSV and reproduce the baseline

# --- which data goes in -------------------------------------------------------------
TEMPLATE_GROUPS = ["3t3", "3t3/human", "k562_DNA"]   # drop entries to hold template fixed
MIN_EVENT_PURITY = 0.0        # pre-filter; v7 additionally enforces >= 0.7 internally
MIN_READ_IDENTITY = 0.0       # Mean_Identity_To_Inferred_Ref; 0.95 is the interesting cut
PURITY_EQ_ONE = True          # v7's own flag: keep only Event_Purity_Score == 1

RUN_TAG = "v7_rerun_baseline"  # names the output folder; change per experiment
MAX_EPOCHS = 180               # v7 default; lower to smoke-test

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import shutil
import numpy as np
import pandas as pd

SAVE_DIR = f"{DRIVE_ROOT}/v7_runs/{RUN_TAG}"
WORK_CSV = f"/content/train_{RUN_TAG}.csv"
os.makedirs(SAVE_DIR, exist_ok=True)

missing = [p for p in [V7_SCRIPT, BASE_MODEL] if not os.path.exists(p)]
if USE_ORGANISED:
    missing += [p for p in [f"{ORGANISED}/metadata.csv", f"{ORGANISED}/curves.npz"]
                if not os.path.exists(p)]
elif not os.path.exists(FALLBACK_CSV):
    missing.append(FALLBACK_CSV)
if missing:
    raise SystemExit("Missing on Drive:\n  " + "\n  ".join(missing))
print("all prerequisites present")

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    print(f"GPU: {gpus if gpus else 'NONE -- switch Runtime to GPU or this will take hours'}")
except Exception as e:
    print("tensorflow not importable yet:", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
all prerequisites present
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Build the training CSV

v7's loader wants a CSV with `Real_Amplicon_Ref_Sequence`, `Event_Purity_Score`, and
≥300 integer-named curve columns (`curve_column_mode = "all_numeric"`). The 301-point
70–100 °C `norm_70_100` grid is what v7 was trained on.

v7 then applies its own filters on top: `Event_Purity_Score >= 0.7`, no `N` in the
sequence, optional `== 1`, de-duplicate by sequence, and drop sequences longer than
`397 - 23 - 24 = 350`. The filters here compose with those, they do not replace them.

In [ ]:
if USE_ORGANISED:
    meta = pd.read_csv(f"{ORGANISED}/metadata.csv", low_memory=False)
    z = np.load(f"{ORGANISED}/curves.npz")
    C = z["norm_70_100"]
    assert len(meta) == len(C), "metadata and curves are not row-aligned"

    keep = (meta["Real_Amplicon_Ref_Sequence"].notna().to_numpy()
            & np.isfinite(C).all(1)
            & meta["template_type"].isin(TEMPLATE_GROUPS).to_numpy()
            & (meta["Event_Purity_Score"].fillna(0) >= MIN_EVENT_PURITY).to_numpy()
            & (meta["Mean_Identity_To_Inferred_Ref"].fillna(0) >= MIN_READ_IDENTITY).to_numpy())
    print(f"{keep.sum()} of {len(meta)} wells pass the pre-filter")
    print(meta[keep].groupby("template_type").agg(
        wells=("well", "size"), purity=("Event_Purity_Score", "mean"),
        identity=("Mean_Identity_To_Inferred_Ref", "mean")).round(3).to_string())

    out = meta.loc[keep, [c for c in meta.columns if not str(c).isdigit()]].reset_index(drop=True)
    curves = pd.DataFrame(C[keep], columns=[str(i) for i in range(C.shape[1])])
    pd.concat([out, curves], axis=1).to_csv(WORK_CSV, index=False)
    print(f"\nwrote {WORK_CSV}  rows={keep.sum()}  curve_cols={C.shape[1]}")
else:
    shutil.copy(FALLBACK_CSV, WORK_CSV)
    print(f"using the original training CSV: {FALLBACK_CSV}")

chk = pd.read_csv(WORK_CSV, low_memory=False, nrows=5)
ncur = len([c for c in chk.columns if str(c).isdigit()])
assert ncur >= 300, f"only {ncur} numeric curve columns; v7 requires >= 300"
assert "Real_Amplicon_Ref_Sequence" in chk.columns and "Event_Purity_Score" in chk.columns
print(f"schema OK — {ncur} curve columns")

12312 of 17151 wells pass the pre-filter
               wells  purity  identity
template_type                         
3t3             6334   0.567     0.856
3t3/human        949   0.490     0.840
k562_DNA        5029   0.692     0.866

wrote /content/train_v7_rerun_baseline.csv  rows=12312  curve_cols=301
schema OK — 301 curve columns


## Load the v7 script and override its config

`run()` reads the module-level singleton `CFG`, so setting attributes on it before
calling `run()` is enough — the script itself is left untouched.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("v7mod", V7_SCRIPT)
v7 = importlib.util.module_from_spec(spec)
sys.modules["v7mod"] = v7
spec.loader.exec_module(v7)          # safe: run() sits behind `if __name__ == "__main__"`

cfg = v7.CFG
cfg.data_path = cfg.local_data_path = WORK_CSV
cfg.model_path = cfg.local_model_path = BASE_MODEL
cfg.save_dir = cfg.local_save_dir = SAVE_DIR
cfg.purity_eq_one = PURITY_EQ_ONE
cfg.max_epochs = MAX_EPOCHS
cfg.curve_column_mode = "all_numeric"

print(f"data      {cfg.data_path}")
print(f"model     {cfg.model_path}")
print(f"save_dir  {cfg.save_dir}")
print(f"epochs    {cfg.max_epochs}   purity_eq_one={cfg.purity_eq_one}   "
      f"variant={cfg.curve_encoder_variant}   embed={cfg.embed_dim}   temp={cfg.temperature}")

# what v7's own loader keeps, before training starts
_, _, _, info = v7.load_data(cfg)
print(f"\nv7 will train on {info['rows']} rows, {info['curve_col_count']} curve columns "
      f"({info['curve_col_first']}..{info['curve_col_last']})")
if info["rows"] < 500:
    print("WARNING: very few rows survive — check TEMPLATE_GROUPS and the purity cuts")

data      /content/train_v7_rerun_baseline.csv
model     /content/drive/Shareddrives/Kelvin Genomics share/traning_data/my_contrastive_model_withflanking0430.keras
save_dir  /content/drive/Shareddrives/Kelvin Genomics share/traning_data/v7_runs/v7_rerun_baseline
epochs    180   purity_eq_one=True   variant=transition_multiscale   embed=64   temp=0.07

v7 will train on 4124 rows, 301 curve columns (0..300)


## Train

~180 epochs. Weights and `..._results.json` are written to `SAVE_DIR` on Drive as it
goes, so an interrupted session still leaves the best checkpoint behind.

In [ ]:
v7.run()

Config:
{
  "seed": 42,
  "split_seed": 42,
  "model_path": "/content/drive/Shareddrives/Kelvin Genomics share/traning_data/my_contrastive_model_withflanking0430.keras",
  "data_path": "/content/train_v7_rerun_baseline.csv",
  "save_dir": "/content/drive/Shareddrives/Kelvin Genomics share/traning_data/v7_runs/v7_rerun_baseline",
  "local_model_path": "/content/drive/Shareddrives/Kelvin Genomics share/traning_data/my_contrastive_model_withflanking0430.keras",
  "local_data_path": "/content/train_v7_rerun_baseline.csv",
  "local_save_dir": "/content/drive/Shareddrives/Kelvin Genomics share/traning_data/v7_runs/v7_rerun_baseline",
  "curve_column_mode": "all_numeric",
  "curve_start_col": 30,
  "curve_end_col_exclusive": 210,
  "expected_min_curve_cols": 300,
  "adapter_before": "CCTACACGACGCTCTTCCGATCT",
  "adapter_after": "AGATCGGAAGAGCACACGTCTGAA",
  "max_seq_len": 397,
  "embed_dim": 64,
  "curve_encoder_variant": "transition_multiscale",
  "curve_encoder_dropout": 0.08,
  "enable_cur

## Results

In [ ]:
res_path = f"{SAVE_DIR}/optimized_minimal_transfer_v7_raw_curve_msa_aux_results.json"
res = json.load(open(res_path))
t = res["fold_results"][0]["test_bank_test"]
print(f"=== {RUN_TAG} ===")
print(f"  test bank n={res['config']['test_size']}")
for k in ("r1", "r5", "r10", "r25"):
    print(f"  R@{k[1:]:<3s} {t[k]:.4f}")
print(f"  MRR   {t.get('mrr', float('nan')):.4f}")

BASELINE = {"r1": 0.1800, "r5": 0.5267, "r10": 0.7133, "r25": 0.9400}
print("\n  vs the published v7 baseline (3,883 rows, purity-filtered, 3t3+k562):")
for k, b in BASELINE.items():
    d = t[k] - b
    print(f"    R@{k[1:]:<3s} {t[k]:.4f}  baseline {b:.4f}  {d:+.4f} ({round(d*150):+d} queries)")

pd.DataFrame([dict(run=RUN_TAG, rows=info["rows"], groups=",".join(TEMPLATE_GROUPS),
                   min_purity=MIN_EVENT_PURITY, min_identity=MIN_READ_IDENTITY,
                   purity_eq_one=PURITY_EQ_ONE, **{f"R@{k[1:]}": t[k] for k in BASELINE})
              ]).to_csv(f"{SAVE_DIR}/summary.csv", index=False)
print(f"\nwrote {SAVE_DIR}/summary.csv")

=== v7_rerun_baseline ===
  test bank n=150
  R@1   0.1933
  R@5   0.5200
  R@10  0.6733
  R@25  0.9133
  MRR   0.3382

  vs the published v7 baseline (3,883 rows, purity-filtered, 3t3+k562):
    R@1   0.1933  baseline 0.1800  +0.0133 (+2 queries)
    R@5   0.5200  baseline 0.5267  -0.0067 (-1 queries)
    R@10  0.6733  baseline 0.7133  -0.0400 (-6 queries)
    R@25  0.9133  baseline 0.9400  -0.0267 (-4 queries)

wrote /content/drive/Shareddrives/Kelvin Genomics share/traning_data/v7_runs/v7_rerun_baseline/summary.csv


## Suggested runs

Change `RUN_TAG` each time so nothing is overwritten, and vary one thing at a time.

| RUN_TAG | settings | question |
|---|---|---|
| `v7_rerun_baseline` | `USE_ORGANISED=False` | does the pipeline reproduce R@1 = 0.18? |
| `v7_all_templates` | all three groups, `PURITY_EQ_ONE=True` | does adding the human plates help? |
| `v7_identity95` | `MIN_READ_IDENTITY=0.95` | do cleaner labels beat more rows? |
| `v7_k562_only` | `TEMPLATE_GROUPS=["k562_DNA"]` | is one template better than a mix? |

**Run the baseline first.** If it does not land near 0.18, something in the data path
has changed and the other runs are not interpretable.

`v7_identity95` versus `v7_all_templates` is the purity question, but note it confounds
quality with quantity — the identity cut also shrinks the training set. Notebook 03's
`all_matched_n` arm is what separates the two, so treat these as a coarse first look and
the ladder as the actual test.